# Waterfilling Algorithm Testing and Visualization
This notebook tests the waterfilling algorithm across different network contexts: Opera, Shale, Sirius, and generic timeslots. It also visualizes the power allocation and identifies bottlenecks.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from Waterfilling_Alg import waterfilling
from Shale_Alg import RR2, RR2_path
from Common_Alg import generate_random_latin_square
from Sirius import generate_full_system
import os

if not os.path.exists('plots'):
    os.makedirs('plots')

In [ ]:
def visualize_waterfilling(channels, total_power, title="Waterfilling Results", filename=None):
    """
    Visualizes the waterfilling power allocation using a stacked bar chart.
    Highlights bottlenecks (channels with 0 allocated power due to high noise).
    Saves the image if a filename is provided.
    """
    channels = np.array(channels)
    
    # Handle 2D case (multiple timeslots)
    if channels.ndim == 2:
        num_timeslots = channels.shape[0]
        # Handle scalar power vs per-timeslot power
        if np.isscalar(total_power):
            powers = [total_power] * num_timeslots
        else:
            powers = total_power
            
        for t in range(num_timeslots):
            # Recursively call for each timeslot
            ts_filename = f"plots/{filename}_ts{t}.png" if filename else None
            visualize_waterfilling(channels[t], powers[t], title=f"{title} - Timeslot {t}", filename=ts_filename)
        return

    # --- 1D Logic ---
    allocation = waterfilling(channels, total_power)
    n = len(channels)
    indices = np.arange(n)
    
    active_mask = allocation > 1e-9
    bottleneck_mask = ~active_mask
    
    # Calculate Water Level (Noise + Allocated Power) for active channels
    if np.any(active_mask):
        # All active channels should sum to the same level
        water_levels = channels[active_mask] + allocation[active_mask]
        water_level = np.mean(water_levels) 
    else:
        water_level = 0 

    plt.figure(figsize=(10, 6))
    
    # 1. Plot Active Noise (Gray)
    if np.any(active_mask):
        plt.bar(indices[active_mask], channels[active_mask], 
                label='Noise (Active)', color='lightgray', edgecolor='black')
        
    # 2. Plot Bottleneck Noise (Red/Salmon) - These are the bottlenecks!
    if np.any(bottleneck_mask):
        plt.bar(indices[bottleneck_mask], channels[bottleneck_mask], 
                label='Noise (Bottleneck)', color='salmon', edgecolor='black', hatch='//')

    # 3. Plot Allocated Power (Blue)
    if np.any(active_mask):
        plt.bar(indices[active_mask], allocation[active_mask], bottom=channels[active_mask], 
                label='Allocated Power', color='skyblue', edgecolor='black')
    
    # 4. Water Level Line
    plt.axhline(y=water_level, color='blue', linestyle='--', linewidth=2, label=f'Water Level ({water_level:.2f})')
    
    plt.xlabel('Channel / Link Index')
    plt.ylabel('Power / Noise Level')
    plt.title(title)
    plt.xticks(indices, [f'Ch {i}' for i in indices])
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    if filename:
        plt.savefig(filename)
        print(f"Saved plot to {filename}")
    plt.show()

## 1. Opera Context Waterfilling

In [ ]:
print("=== Testing Waterfilling with Opera Context ===")
rack_weights = [5, 10, 5, 20, 100, 5, 10, 5]
total_power = 50
allocation = waterfilling(rack_weights, total_power)
print(f"Rack Weights: {rack_weights}")
print(f"Power Allocation: {allocation}")
visualize_waterfilling(rack_weights, total_power, title="Opera Waterfilling Bottlenecks", filename="opera_wf.png")

## 2. Shale Context Waterfilling

In [ ]:
print("=== Testing Waterfilling with Shale Context ===")
adj_matrix = RR2(3, 2)
node_0_links = adj_matrix[0]
active_links = [x for x in node_0_links if x is not None]
num_links = len(active_links)
random.seed(42)
link_noise = [random.randint(1, 20) for _ in range(num_links)]
total_power = 30
visualize_waterfilling(link_noise, total_power, title="Shale Waterfilling Bottlenecks", filename="shale_wf.png")

## 3. Multiple Timeslots Waterfilling

In [ ]:
print("=== Testing Waterfilling with Multiple Timeslots ===")
channels = [
    [5, 5, 5, 5],
    [20, 20, 20, 20],
    [5, 20, 5, 20]
]
total_power = 20
visualize_waterfilling(channels, total_power, title="Timeslot Waterfilling", filename="timeslots_wf")

## 4. Sirius Context Waterfilling

In [ ]:
print("=== Testing Waterfilling with Sirius Context ===")
wavelengths = 3
ports = 2
nodes = 6
As, Ws, P_mat = generate_full_system(wavelengths, ports, nodes)
num_timeslots = len(Ws)
channels_sirius = []
random.seed(100)
for t, W in enumerate(Ws):
    timeslot_noise = [random.randint(1, 20) for _ in range(nodes)]
    channels_sirius.append(timeslot_noise)
total_power_sirius = 30
visualize_waterfilling(channels_sirius, total_power_sirius, title="Sirius Waterfilling Bottlenecks", filename="sirius_wf")

## 5. Performance Comparison

In [ ]:
print("=== Comparing Waterfilling Performance ===")
opera_noise = [5, 10, 5, 20, 100, 5, 10, 5]
shale_noise = [4, 1, 9, 8]
sirius_noise = [5, 15, 15, 6, 13, 12]
generic_noise = [5, 5, 5, 5]
power_levels = np.linspace(10, 100, 10)
results = {"Opera": [], "Shale": [], "Sirius": [], "Generic": []}
scenarios = [("Opera", opera_noise), ("Shale", shale_noise), ("Sirius", sirius_noise), ("Generic", generic_noise)]
for label, noises in scenarios:
    for P in power_levels:
        alloc = np.array(waterfilling(noises, P))
        noise_arr = np.array(noises)
        snr = alloc / noise_arr
        capacity = np.sum(np.log2(1 + snr))
        results[label].append(capacity)

plt.figure(figsize=(10, 6))
for label, capacities in results.items():
    plt.plot(power_levels, capacities, marker='o', label=label)
plt.xlabel("Total Power Budget (P)")
plt.ylabel("Capacity (Shannon Sum Rate)")
plt.title("Waterfilling Performance Comparison")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig("plots/performance_comparison.png")
plt.show()